In [8]:
from typing import (
    TypedDict,
    Literal,
    Annotated,
)

import operator
import os
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from langchain_core.tools import tool
# from langchain.agents import create_agent

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.types import (
    Command,
    Send,
)
from langgraph.checkpoint.memory import InMemorySaver

In [9]:
load_dotenv()

True

In [10]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found - add it to your .env file.")
    
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    max_retries=3,   # free models hit 429 rate limits often
    timeout=120,
)

In [11]:
# agent state

class AgentState(TypedDict):
    Task: str
    message: Annotated[list, add_messages]
    output: str
    route: str

In [12]:
# creating Specialist agents for Frontend teams

# react agent
def react_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a React specialist. Handle all React component and frontend tasks.")
    response = model.invoke(system, state["message"])
    return {"messages": [response], "output": response.content}

# ui agent
def ui_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a UI specialist. Build clean, accessible user interfaces and handle all UI releated task.")
    response = model.invoke(system, state["message"])
    return {"message": response, "output": response.content}

# Accessibility Agent
def accessibility_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Accessibility specialist. Ensure all UI components meet WCAG 2.1 AA standards: semantic HTML, keyboard navigation, screen reader support, color contrast, and proper ARIA usage.")
    response = model.invoke(system, state["message"])
    return {"message": response, "output": response.content}

In [13]:
# creating Specialist agents for backend teams

# api agent
def api_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an API specialist. Design RESTful APIs, handle routing, validation, serialization, and ensure consistent error responses.")
    response = model.invoke(system, state["message"])
    return {"message":[response], "output": response.content}

# auth agent
def auth_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Authentication specialist. Implement secure auth: JWT/OAuth2, token refresh, password hashing, MFA, session management, and RBAC.")
    response = model.invoke(system, state["message"])
    return {"message":[response], "output": response.content}

# microsoft agent
def microsoft_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a Microsoft ecosystem specialist. Handle Azure services, Microsoft Graph, Entra ID, Power Platform, and Office 365 integrations.")
    response = model.invoke(system, state["message"])
    return {"messages": [response], "output": response.content}

In [14]:
# creating Specialist agents for database teams

# database agents
def sql_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL specialist. Help with database and SQL tasks by producing accurate, efficient, and secure queries. Use only the schema and requirements provided; if essential details are missing, ask a concise clarifying question. Briefly explain assumptions when needed, and return only the requested result.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}

def schema_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert database schema designer. Help create, review, and improve database schemas based on the requirements provided. Recommend appropriate tables, columns, data types, primary and foreign keys, constraints, and relationships. Prioritize data integrity, clarity, and maintainability. Ask a concise clarifying question if essential requirements are missing, and return only the requested result.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}

def Query_optimization_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL query optimization specialist. Analyze SQL queries and the provided schema to identify ways to improve performance without changing the results. Consider indexes, joins, filtering, query plans, and database-specific behavior. State any assumptions, ask a concise clarifying question if essential details are missing, and return only the requested result.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}

In [ ]:
# creating Specialist agents for QA teams

# unit test Agent
def Unit_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert unit testing specialist. Help write focused tests for individual functions, methods, and components. Cover expected behavior, edge cases, and error conditions using the project’s existing testing framework and conventions. Keep tests deterministic and independent. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}
# Integration test Agent
def integration_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert integration testing specialist. Help test how multiple components, services, or modules work together. Follow the project’s existing testing framework and conventions, and cover key data flows, boundaries, and failure cases. Keep tests reliable and avoid unnecessary external dependencies. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}
# regresion Agent
def regression_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert regression testing specialist. Help create tests that catch previously fixed bugs and prevent existing behavior from breaking. Use the provided bug details, expected behavior, and project testing conventions to write focused, repeatable tests. If essential context is missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}

In [ ]:
# creating Specialist agents for DevOps teams

# Docker Agent
def docker_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Docker specialist. Help create, review, and troubleshoot Dockerfiles, images, and Compose configurations. Recommend secure, efficient, and reproducible container practices, following the project’s requirements and conventions. If essential details are missing, ask a concise clarifying question; otherwise return only the requested configuration or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}
# Kubernetes Agent
def kubernetes_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Kubernetes specialist. Help design, write, review, and troubleshoot Kubernetes resources such as Deployments, Services, ConfigMaps, and Ingresses. Prioritize reliability, security, and maintainability, and use the requirements and environment details provided. If essential information is missing, ask a concise clarifying question; otherwise return only the requested manifests or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}
# CI/CD Agent
def cicd_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert CI/CD specialist. Help design, write, and troubleshoot continuous integration and deployment pipelines using the platform and project requirements provided. Focus on reliable automation, clear validation steps, secure handling of secrets, and safe deployments. If essential details are missing, ask a concise clarifying question; otherwise return only the requested pipeline configuration or guidance.")
    response = model.invoke(system, state["message"])
    return {"message": [response], "output": response.content}